# Notebook 01 — EDA & Data Preparation

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** A — Data & EDA (Section 2 below) + C — MLOps / Data Pipeline (everything else)  
**Estimated Runtime:** 2-5 minutes

---

## What this notebook does

1. Loads the crypto scam dataset and uploads the raw file to S3
2. Exploratory Data Analysis — **placeholder, Royston's section (Focus A)**
3. Runs the data pipeline inline (text cleaning, engineered indicator features, TF-IDF, stratified train/test split) — self-contained like the tutor's notebook, so no repo clone / `utils` import is needed to run this in SageMaker Studio
4. Saves the processed train/test data and the fitted TF-IDF vectorizer to S3 for Notebook 02/03

## Dataset

- Local copy: `data/crypto_scam_dataset.csv` (already in this repo)
- Source: [Kaggle — Crypto Scam Dataset](https://www.kaggle.com/datasets/theeyanyan/crypto-scam-dataset/data), as cited in the project proposal
- 10,000 rows, columns `id`, `platform`, `text`, `label` (`scam` / `legit`), no missing values

> **Adapted for team03 (Ong Hui Lin, Student 2 — MLOps & Deployment) from the ITI113 course template notebook `01_eda_and_data_preparation.ipynb`.** The tutor's notebook mixes two things: EDA/insights (individually graded as Focus A — Royston's) and data loading/preparation (the MLOps data pipeline — this project's job). **Section 2 (EDA) is intentionally left as a placeholder for Royston to fill in** rather than written on his behalf. Everything else follows the tutor's structure as closely as the data allows, adapted to text data (TF-IDF) instead of tabular clinical features. Data cleaning and feature engineering are written **inline in this notebook**, the same way the tutor's own notebook is self-contained, rather than imported from `utils/preprocessing.py` — this notebook now runs standalone in SageMaker Studio with no repo clone required. Sections marked **`# ADAPTED`** explain what had to change.
>
> **Trade-off:** `utils/preprocessing.py` (used by Notebook 02 and the Streamlit app) has this same cleaning/feature logic. Keeping this notebook's inline copy and that file in sync is now a manual step — if the cleaning or feature-engineering logic ever changes, update both places, or the app's predictions could drift from what the model was actually trained on.

In [1]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade --force-reinstall "sagemaker>=2,<3" boto3 botocore
!pip install -q -U 'kagglehub'

  Using cached sagemaker-2.257.5-py3-none-any.whl.metadata (19 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)


  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)


  Using cached docker-7.2.0-py3-none-any.whl.metadata (3.8 kB)


  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 61/62 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 61/62 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 61/62 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62/62 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
dash 2.18.1 requires dash-core-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-html-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-table==5.0.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
aiobotocore 3.7.0 requires botocore<1.43.1,>=1.42.90, but you have botocore 1.43.62 which is incompatible.
amazon-sagemaker-sql-editor 0.2.6 requires

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

Edit the values in this cell. Everything else references these variables.

In [2]:
import sagemaker, boto3

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

TEAM_ID = "team03"
STUDENT_ID = "s301"

BUCKET = "nyp-26s1-iti113"
PROJECT_NAME = "crypto-scam-detector"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"  # matches the PREFIX pattern used in Notebook 02

RANDOM_STATE = 42   # matches utils/preprocessing.py's default, kept identical so results match your Mac runs
TEST_SIZE    = 0.20 # matches utils/preprocessing.py's default

print(f"Bucket     : {BUCKET}")
print(f"Prefix     : {PREFIX}")
print(f"Region     : {region}")
print(f"Role       : {role.split('/')[-1]}")
print(f"Team ID    : {TEAM_ID}")
print(f"Student ID : {STUDENT_ID}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Bucket     : nyp-26s1-iti113
Prefix     : iti113/team03/data/crypto-scam-detector
Region     : ap-southeast-1
Role       : SageMakerExecutionRole-ITI113-Team03
Team ID    : team03
Student ID : s301


## 1. Load Dataset

**ADAPTED:** the tutor's version downloads the UCI Heart Disease dataset from a public URL, then uploads it to S3 for version control. This project's raw dataset doesn't have an equivalent no-auth public URL, so `kagglehub` downloads it directly from Kaggle instead -- this avoids depending on a local repo checkout of `data/crypto_scam_dataset.csv`, which matters in SageMaker Studio (no repo clone here). This cell checks S3 first and loads from there if the raw file already exists (`{PREFIX}/raw/crypto_scam_dataset.csv`); only on the very first run anywhere does it fall back to `kagglehub` and upload the result -- so every run after that, on any machine, reads from S3 instead of re-fetching from Kaggle each time.

| Column | Description |
|--------|-------------|
| id | Unique message identifier |
| platform | Source platform (Telegram, X/Twitter, SMS, Email, Discord, Reddit) |
| text | Message content |
| label | `scam` or `legit` |

In [3]:
import io
import os

import kagglehub
import pandas as pd

s3 = boto3.client('s3')
RAW_S3_KEY = f'{PREFIX}/raw/crypto_scam_dataset.csv'
RAW_S3_URI = f's3://{BUCKET}/{RAW_S3_KEY}'

try:
    obj = s3.get_object(Bucket=BUCKET, Key=RAW_S3_KEY)
    df_raw = pd.read_csv(io.BytesIO(obj['Body'].read()))
    print(f'Loaded raw dataset from S3: {RAW_S3_URI}')
except s3.exceptions.NoSuchKey:
    dataset_path = kagglehub.dataset_download("theeyanyan/crypto-scam-dataset")
    csv_file = os.path.join(dataset_path, "crypto_scam_dataset.csv")
    df_raw = pd.read_csv(csv_file)

    buf = io.StringIO()
    df_raw.to_csv(buf, index=False)
    s3.put_object(Bucket=BUCKET, Key=RAW_S3_KEY, Body=buf.getvalue())
    print(f'Not yet in S3 -- downloaded via kagglehub and uploaded to: {RAW_S3_URI}')

print(f'Shape : {df_raw.shape}')
print(f'Label : {df_raw["label"].value_counts().to_dict()}')
df_raw.head()

Loaded raw dataset from S3: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/raw/crypto_scam_dataset.csv
Shape : (10000, 4)
Label : {'scam': 5500, 'legit': 4500}


,id,platform,text,label
0,1,Email,Following up on your recovery request for wall...,scam
1,2,SMS,Does anyone have a good resource for understan...,legit
2,3,Telegram,The MetaSwap team posted a security disclosure...,legit
3,4,Telegram,Been dollar-cost averaging into ETH for about ...,legit
4,5,SMS,Your account requires immediate verification d...,scam


## 2. Exploratory Data Analysis

**Placeholder — Royston's section (Focus A).** The Progress Check rubric wants completed EDA with insights and patterns identified here: missing-value checks (none, per the load above, but worth confirming), class balance (`scam` vs `legit`), platform distribution, message length distribution, common scam-language patterns, and any data-quality issues or bias observations for the AI Governance checklist. Left empty intentionally rather than filled in on his behalf — see `df_raw` from Section 1 above as the starting point.

## 3. Feature Engineering & Preprocessing

**ADAPTED — now fully inline**, matching the tutor's own notebook (their `age_group`/`high_risk_count` engineering, split, and `StandardScaler` are inline too, specific to the tabular heart-disease features). This project's text data needs different steps instead: text cleaning, the proposal's engineered indicator features (urgency, contact/link, structural characteristics), a stratified train/test split, and TF-IDF fitting — written inline here rather than imported from `utils/preprocessing.py`, so this notebook runs standalone with no repo clone / `utils` import required. No feature scaling is applied, matching the rest of the project (TF-IDF features aren't scaled elsewhere either).

In [4]:
# Step 1: clean text (mirrors utils/preprocessing.py's clean_text())
import re

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text: str) -> str:
    """Normalise raw message text before TF-IDF vectorisation."""
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df = df_raw.copy()
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,Following up on your recovery request for wall...,following up on your recovery request for wall...
1,Does anyone have a good resource for understan...,does anyone have a good resource for understan...
2,The MetaSwap team posted a security disclosure...,the metaswap team posted a security disclosure...
3,Been dollar-cost averaging into ETH for about ...,been dollar-cost averaging into eth for about ...
4,Your account requires immediate verification d...,your account requires immediate verification d...


In [5]:
# Step 2: engineered indicator features (mirrors utils/indicators.py + utils/preprocessing.py)
# Keyword lists come straight from the project proposal's indicator design.
URGENT_KEYWORDS = [
    "urgent", "immediately", "act now", "act fast", "hurry", "limited time",
    "today only", "expires today", "offer ends soon", "last chance",
    "don't miss out", "within 24 hours", "within 1 hour", "respond now",
    "claim now", "limited slots", "before it's too late", "time-sensitive",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed return", "guaranteed returns", "guaranteed profit",
    "risk-free", "risk free", "100% profit", "double your money",
    "high returns", "\u7a33\u8d5a\u4e0d\u8d54",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer funds", "send payment", "pay now", "top up",
    "bitcoin", "btc", "ethereum", "eth", "usdt", "wallet address",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "wechat", "private chat",
    "dm me", "direct message",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "wallet password", "recovery phrase",
    "otp", "verification code", "security code",
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches(message: str, keywords: list) -> list:
    message_lower = message.lower()
    return [k for k in keywords if k.lower() in message_lower]

def extract_engineered_features(text: str) -> dict:
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "wallet_address_count": len(wallet_matches),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "message_length": len(text),
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
        "digit_count": digit_count,
    }

engineered = pd.DataFrame([extract_engineered_features(t) for t in df["text"]], index=df.index)
ENGINEERED_COLS = list(engineered.columns)
df = pd.concat([df, engineered], axis=1)
df[ENGINEERED_COLS].head()

,urgency_keyword_count,guaranteed_return_keyword_count,countdown_phrase_count,exclamation_count,urgency_score,has_wallet_address,wallet_address_count,has_url,url_count,has_email,has_phone_number,payment_keyword_count,off_platform_keyword_count,credential_keyword_count,message_length,capital_letter_ratio,has_numeric_content,digit_count
0,1,0,1,0,2,1,2,0,0,0,1,1,0,0,292,0.0370,1,61
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,212,0.0412,1,4
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,176,0.0284,0,0
3,0,0,0,0,0,0,0,0,0,0,1,1,0,0,286,0.0327,1,12
4,2,0,1,0,3,0,0,0,0,1,0,0,0,0,220,0.0718,1,3


In [6]:
# Step 3: train/test split (stratified on label)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

print(f'Train: {train_df.shape[0]} rows  |  Test: {test_df.shape[0]} rows')
print(f'Train label balance: {(train_df["label"]=="scam").mean():.3f}  '
      f'(full dataset: {(df_raw["label"]=="scam").mean():.3f})')
print(f'Test  label balance: {(test_df["label"]=="scam").mean():.3f}  (stratification confirmed)')

Train: 8000 rows  |  Test: 2000 rows
Train label balance: 0.550  (full dataset: 0.550)
Test  label balance: 0.550  (stratification confirmed)


In [7]:
# Step 4: fit TF-IDF vectorizer -- IMPORTANT: fit on TRAIN text only, never on test
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
vectorizer.fit(train_df["clean_text"])

print(f'TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}')

TF-IDF vocabulary size: 5000


## 4. Save Processed Data to S3

In [8]:
# ADAPTED: the tutor's version saves four numeric CSVs (train/test features and labels
# split apart), which fits tabular data. This project's processed output is train_df /
# test_df (text + engineered features + label together, matching data/processed/*.csv
# already produced locally) plus the fitted TF-IDF vectorizer -- so three artifacts are
# uploaded instead. If Notebook 02/03 are later switched to load from S3 instead of
# re-running the pipeline locally, they should read these three, not the tutor's four.

import joblib
import tempfile

def save_csv_to_s3(df, bucket, key):
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    return f's3://{bucket}/{key}'

def save_joblib_to_s3(obj, bucket, key):
    with tempfile.NamedTemporaryFile(suffix='.joblib') as tmp:
        joblib.dump(obj, tmp.name)
        s3.upload_file(tmp.name, bucket, key)
    return f's3://{bucket}/{key}'

p = f'{PREFIX}/processed'
paths = {
    'train'            : save_csv_to_s3(train_df, BUCKET, f'{p}/train.csv'),
    'test'             : save_csv_to_s3(test_df, BUCKET, f'{p}/test.csv'),
    'tfidf_vectorizer' : save_joblib_to_s3(vectorizer, BUCKET, f'{p}/tfidf_vectorizer.joblib'),
}
for name, uri in paths.items():
    print(f'{name:<20}: {uri}')

PROCESSED_PREFIX = f's3://{BUCKET}/{p}'
print(f'\nProcessed prefix: {PROCESSED_PREFIX}')
print('Next: open Notebook 02 to run baseline experiments with MLflow.')

train               : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/processed/train.csv
test                : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/processed/test.csv
tfidf_vectorizer    : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/processed/tfidf_vectorizer.joblib

Processed prefix: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/processed
Next: open Notebook 02 to run baseline experiments with MLflow.


---
## Checklist before Notebook 02

- [ ] Raw dataset uploaded to S3
- [ ] EDA completed by Royston (Section 2) — insights, class balance, platform distribution, data quality / bias observations for AI Governance
- [ ] Data pipeline run successfully (inline, no `utils` import) — clean text, engineered indicator features, TF-IDF fitted, stratified train/test split
- [ ] Processed train/test data and TF-IDF vectorizer saved to S3
- [ ] EDA observations and governance flags documented for the checklist